# Lab 10: RNA-seq Data Analysis

**Dataset:** GSE96870 — mouse brain RNA-seq, control (day 0) vs. cerebellum, 6 samples (3 replicates each). All datasets and intermediate results are available under shared folder on the Silo server under /nobackup/yye/Bio2/datasets/rnaseq. 

**count matrix:** Count table from Part I; will be used for Part II. This file is available under the rnaseq data folder; also available on github. 

**Reference:** Mouse GRCm39, chromosome 19 only  

**Estimated time:** Depending. It takes many hours to produce the gene-sample count table from the RNAseq files (Parts 1 steps on the Linux server). Analysis and visualization (Part II) will be quicker. 

---

## Workflow overview

We will use a linux server (the Silo server) for the first few steps (Part I). 

After deriving the feature table, we will use jupyter notebook (VS Code on your own laptop) for anaysis and visualization (Part II). 


| Part | Where | Steps |
|------|-------|-------|
| 1 | Linux server | from RNAseq data to count matrix |
| 1.1 | Linux server | Quality control (FastQC, Trim Galore, MultiQC) |
| 1.2 | Linux server | Read alignment (STAR) |
| 1.3 | Linux server | Count table (featureCounts) |
| 2 | **This notebook** | Analysis of the count matrix |
| 2.1 | **This notebook** | Differential expression (pydeseq2) |
| 2.2 | **This notebook** | Gene set enrichment analysis (gseapy) |

---

## Parts 1: Linux server (command-line reference)

> Run these steps on the server **before** opening this notebook. They are shown here for reference.
> Students are welcome to try these steps on the Silo server, but they won't be able to finish the steps in the lab.  
> Students can try Part 1.3, to create feature table, using precalculated reads mapping results. 

### Setup

A conda environment (bio2-rnaseq) is created on Silo. Datasets (RNAseq and reference genome) are also available on the server. 

```bash
conda activate /nobackup/yye/Bio2/miniconda/envs/bio2-rnaseq 
mkdir -p lab10_rnaseq/{raw_data,trimmed,genome,index,aligned,counts,qc}
cd lab10_rnaseq
export BASE=$(pwd)
```

### GSE96870 reads (1 M reads per sample for speed) -- Downloaded and availble on Silo

| SRA accession | Condition | Replicate |
|---------------|-----------|----------|
| SRR5454079 | control (day 0) | 1 |
| SRR5454080 | control (day 0) | 2 |
| SRR5454081 | control (day 0) | 3 |
| SRR5454082 | cerebellum | 1 |
| SRR5454083 | cerebellum | 2 |
| SRR5454084 | cerebellum | 3 |

```bash
for ACC in SRR5454079 SRR5454080 SRR5454081 SRR5454082 SRR5454083 SRR5454084; do
  fasterq-dump $ACC --outdir $BASE/raw_data --threads 4
  gzip $BASE/raw_data/${ACC}.fastq
done

# Rename to meaningful sample names
mv $BASE/raw_data/SRR5454079.fastq.gz $BASE/raw_data/ctrl_rep1.fastq.gz
mv $BASE/raw_data/SRR5454080.fastq.gz $BASE/raw_data/ctrl_rep2.fastq.gz
mv $BASE/raw_data/SRR5454081.fastq.gz $BASE/raw_data/ctrl_rep3.fastq.gz
mv $BASE/raw_data/SRR5454082.fastq.gz $BASE/raw_data/cbl_rep1.fastq.gz
mv $BASE/raw_data/SRR5454083.fastq.gz $BASE/raw_data/cbl_rep2.fastq.gz
mv $BASE/raw_data/SRR5454084.fastq.gz $BASE/raw_data/cbl_rep3.fastq.gz
```

### Download mouse chr19 reference (GRCm39, Ensembl 110)

```bash
wget -P $BASE/genome \
  https://ftp.ensembl.org/pub/release-110/fasta/mus_musculus/dna/Mus_musculus.GRCm39.dna.chromosome.19.fa.gz
wget -P $BASE/genome \
  https://ftp.ensembl.org/pub/release-110/gtf/mus_musculus/Mus_musculus.GRCm39.110.gtf.gz
gunzip $BASE/genome/*.gz
grep -P '^19\t' $BASE/genome/Mus_musculus.GRCm39.110.gtf > $BASE/genome/Mus_musculus.GRCm39.110.chr19.gtf
```

---

### Part 1.1: Quality Control (~10 min) -- skip if no time

```bash
# FastQC on raw reads
mkdir -p $BASE/qc/fastqc_raw
fastqc $BASE/raw_data/*.fastq.gz --outdir $BASE/qc/fastqc_raw --threads 6

# Trim adapters (single-end, reverse-stranded TruSeq library)
for SAMPLE in ctrl_rep1 ctrl_rep2 ctrl_rep3 cbl_rep1 cbl_rep2 cbl_rep3; do
  trim_galore --quality 20 --length 36 --cores 4 \
    --output_dir $BASE/trimmed $BASE/raw_data/${SAMPLE}.fastq.gz
done


> **Checkpoint:** Check html files under qc/fastqc_raw/ for reads passing rates.

---

### Part 1.2: Read Alignment with STAR 
This step is very time consuming
- it takes several hours to complete all six RNAseq data sets; about 50 mins per data set on Silo when using four threads. 
- skip this step if no time
- students can use precalcualted mapping results for the next step.

```bash
# Build STAR index for chr19 (skip if pre-built index provided)
mkdir -p $BASE/index/star_mm39_chr19
STAR \
  --runMode genomeGenerate \
  --genomeDir $BASE/index/star_mm39_chr19 \
  --genomeFastaFiles $BASE/genome/Mus_musculus.GRCm39.dna.chromosome.19.fa \
  --sjdbGTFfile $BASE/genome/Mus_musculus.GRCm39.110.chr19.gtf \
  --genomeSAindexNbases 11 \
  --runThreadN 4

# Align all 6 samples
for SAMPLE in ctrl_rep1 ctrl_rep2 ctrl_rep3 cbl_rep1 cbl_rep2 cbl_rep3; do
  STAR \
    --runMode alignReads \
    --genomeDir $BASE/index/star_mm39_chr19 \
    --readFilesIn $BASE/trimmed/${SAMPLE}_trimmed.fq.gz \
    --readFilesCommand zcat \
    --outSAMtype BAM SortedByCoordinate \
    --outSAMattributes NH HI AS NM \
    --outFileNamePrefix $BASE/aligned/${SAMPLE}_ \
    --runThreadN 4
done

# Index BAMs and check alignment rates
for SAMPLE in ctrl_rep1 ctrl_rep2 ctrl_rep3 cbl_rep1 cbl_rep2 cbl_rep3; do
  samtools index $BASE/aligned/${SAMPLE}_Aligned.sortedByCoord.out.bam
done
grep 'Uniquely mapped reads %' $BASE/aligned/*Log.final.out
```

> **Checkpoint:** Expect 60–80% uniquely mapped reads per sample (lower than whole-genome because only chr19 reads align).

---

### Part 1.3: Count Table Preparation (a quick step) -- do this step using provided reads mapping results if step 1 and 2 were skipped. 

```bash
# Run featureCounts (single-end, reverse-stranded)
featureCounts \
  -T 4 -s 2 \
  -a $BASE/genome/Mus_musculus.GRCm39.110.chr19.gtf \
  -o $BASE/counts/all_samples_counts.txt \
  $BASE/aligned/*_Aligned.sortedByCoord.out.bam

# Build a clean count matrix
tail -n +2 $BASE/counts/all_samples_counts.txt | \
  cut -f1,7- | \
  sed 's|.*/aligned/||g; s|_Aligned.*||g' \
  > $BASE/counts/count_matrix.tsv

head -3 $BASE/counts/count_matrix.tsv
```

> **Checkpoint:** `Assigned` fraction in `all_samples_counts.txt.summary` should be ≥ 60%.

---

### Transfer the count matrix to your laptop and continue below.

---


## Part 2: Analysis of the count matrix
- we will use the count_matrix.tsv derived as above for the following analyses
- differential expression analysis: DESeq2 and plotting (e.g., volcano plot and PCA plot)
- gene set enrichment analysis by GSEApy

### Part 2.1 -- install needed packages
- PyDESeq2 for differential expression analysis
- GSEApy: a comprehensive package for performing gene set enrichment analysis in Python
- mygene: lightweight client wrapper for accessing the MyGene.info REST web services, which provide high-performance gene annotation and query APIs. 

In [102]:
#comment it out once installed
#%pip install seaborn pydeseq2 gseapy mygene

### Part 2.1 Differential expression analysis using DESeq2
To perform differential expression analysis (DEA), PyDESeq2 requires two types of inputs:
- A count matrix of shape ‘number of samples’ x ‘number of genes’, containing read counts (non-negative integers),
- Metadata (or “column” data) of shape ‘number of samples’ x ‘number of variables’, containing sample annotations that will be used to split the data in cohorts.
Both should be provided as pandas dataframes.


In [ ]:
# Cell 1 — Imports and paths
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.utils import load_example_data
from sklearn.decomposition import PCA
print("All libraries imported successfully.")

In [ ]:
# Cell 1 — Set up the files
# Update COUNTS path if your count_matrix.tsv lives elsewhere
COUNTS = os.path.expanduser("./count_matrix.tsv")
OUT    = os.path.expanduser("~/rnaseq_output")
os.makedirs(OUT, exist_ok=True)
print("Paths OK:", os.path.exists(COUNTS))

In [ ]:
# Cell 2.1 — Load counts and metadata
counts = pd.read_csv(COUNTS, sep="\t", index_col=0)

counts = counts.T   # pydeseq2 expects samples as rows

# GSE96870: control (day 0) vs cerebellum
metadata = pd.DataFrame(
    {"condition": ["cerebellum"]*3 + ["ctrl"]*3 },
    index=counts.index
)

print("Count matrix shape:", counts.shape)   # expect (6, ~1000 genes on chr19)
print()
print(metadata)

In [106]:
# Cell 2.2 - apply some filtering, e.g., removing samples for which condition is NaN, and remove genes with low counts
samples_to_keep = ~metadata.condition.isna()
counts = counts.loc[samples_to_keep]
metadata = metadata.loc[samples_to_keep]

min_counts = 10
genes_to_keep = counts.columns[counts.sum(axis=0) >= min_counts]
counts = counts[genes_to_keep]


In [ ]:
# Cell 3 — Build DESeq2 dataset and run differential expression
dds = DeseqDataSet(
    counts=counts,
    metadata=metadata,
    design_factors="condition",
    ref_level=["cerebellum", "ctrl"],
    refit_cooks=True,
)
dds.deseq2()

stat_res = DeseqStats(dds, contrast=("condition", "cerebellum", "ctrl"))
stat_res.summary()

results = stat_res.results_df.dropna(subset=["padj"]).sort_values("padj")
results.to_csv(f"{OUT}/deseq2_results.csv")

n_up   = ((results.padj < 0.05) & (results.log2FoldChange >  1)).sum()
n_down = ((results.padj < 0.05) & (results.log2FoldChange < -1)).sum()
print(f"\nSignificant genes (padj<0.05, |log2FC|>1): {n_up} up, {n_down} down")
results.head(10)

In [ ]:
# Cell 4 — Volcano plot
fig, ax = plt.subplots(figsize=(7, 5))

results["neg_log10_padj"] = -np.log10(results["padj"].clip(lower=1e-300))
results["color"] = "grey"
results.loc[(results.padj < 0.05) & (results.log2FoldChange >  1), "color"] = "firebrick"
results.loc[(results.padj < 0.05) & (results.log2FoldChange < -1), "color"] = "steelblue"

ax.scatter(results.log2FoldChange, results.neg_log10_padj,
           c=results.color, alpha=0.6, s=15)
ax.axhline(-np.log10(0.05), color="black", linestyle="--", linewidth=0.8, label="padj=0.05")
ax.axvline( 1, color="black", linestyle="--", linewidth=0.8)
ax.axvline(-1, color="black", linestyle="--", linewidth=0.8)
ax.set_xlabel("log2 Fold Change (cerebellum / ctrl)")
ax.set_ylabel("-log10(padj)")
ax.set_title("Volcano plot — GSE96870 chr19")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT}/volcano.pdf")
plt.show()
print("Saved: volcano.pdf")

### Question: does it look like a typical volcanol plot? 

In [ ]:
# Cell 5 — PCA plot
norm_counts = dds.layers["normed_counts"]
log_counts  = np.log1p(norm_counts)

pca = PCA(n_components=2)
pc_scores = pca.fit_transform(log_counts)
pct_var = pca.explained_variance_ratio_ * 100

fig, ax = plt.subplots(figsize=(5, 4))
palette = ["steelblue"]*3 + ["firebrick"]*3
for i, (x, y) in enumerate(pc_scores):
    ax.scatter(x, y, color=palette[i], s=80, zorder=3)
    ax.annotate(counts.index[i], (x, y), fontsize=8,
                xytext=(5, 3), textcoords="offset points")

# Manual legend
from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='steelblue', markersize=8, label='ctrl'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='firebrick',  markersize=8, label='cerebellum'),
]
ax.legend(handles=legend_handles, fontsize=8)
ax.set_xlabel(f"PC1 ({pct_var[0]:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pct_var[1]:.1f}% variance)")
ax.set_title("PCA — log1p normalized counts")
plt.tight_layout()
plt.savefig(f"{OUT}/pca.pdf")
plt.show()
print("Saved: pca.pdf")
# Checkpoint: ctrl and cerebellum replicates should separate along PC1.

---

## Part 2.2: Gene Set Enrichment Analysis

gseapy fetches mouse MSigDB Hallmark gene sets automatically — make sure your laptop is connected to the internet.

**Ranking metric:** DESeq2 Wald statistic (encodes both direction and significance). Genes at the top are most upregulated in the cerebellum; genes at the bottom are most downregulated.

In [ ]:
# Cell 1 — Map Ensembl IDs to mouse gene symbols
import gseapy as gp
import mygene

mg = mygene.MyGeneInfo()
mapping = mg.querymany(
    results.index.tolist(),
    scopes="ensembl.gene",
    fields="symbol",
    species="mouse",
    as_dataframe=True
)

results["symbol"] = results.index.map(mapping["symbol"].to_dict())
results_sym = results.dropna(subset=["symbol", "stat"])

# Rank by Wald statistic; average duplicate gene symbols
ranked = (
    results_sym.groupby("symbol")["stat"]
    .mean()
    .sort_values(ascending=False)
)
print(f"Ranked gene list: {len(ranked)} genes")
print(ranked.head())

In [ ]:
# Cell 2 — Run pre-ranked GSEA (Hallmark gene sets)
gsea_res = gp.prerank(
    rnk=ranked,
    gene_sets="MSigDB_Hallmark_2020",
    outdir=f"{OUT}/gsea_out",
    min_size=5,
    max_size=500,
    permutation_num=200,   # increase to 1000 for publication-quality results
    seed=42,
    verbose=False,
)

gsea_df = gsea_res.res2d.sort_values("FDR q-val")
gsea_df.to_csv(f"{OUT}/gsea_results.csv", index=False)
gsea_df[["Term", "NES", "NOM p-val", "FDR q-val"]].head(10)

In [ ]:
# Cell 3 — Dot plot of significant pathways (FDR < 0.25)
sig = gsea_df[gsea_df["FDR q-val"] < 0.25].copy()
if sig.empty:
    print("No pathways reach FDR < 0.25 on the chr19 subset. Try FDR < 0.5 or the full dataset.")
else:
    sig["-log10_FDR"] = -np.log10(sig["FDR q-val"].clip(lower=1e-10))
    sig["direction"]  = sig["NES"].apply(lambda x: "Up in cerebellum" if x > 0 else "Down in cerebellum")
    sig = sig.sort_values("NES")

    fig, ax = plt.subplots(figsize=(8, 0.45 * max(len(sig), 4) + 1))
    scatter = ax.scatter(
        sig["NES"], sig["Term"],
        c=sig["direction"].map({"Up in cerebellum": "firebrick", "Down in cerebellum": "steelblue"}),
        s=sig["-log10_FDR"] * 40,
        alpha=0.85
    )
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Normalized Enrichment Score")
    ax.set_title("GSEA Hallmark pathways — GSE96870 chr19 (FDR < 0.25)")

    # Size legend
    for s_val in [1, 2, 3]:
        ax.scatter([], [], c="grey", alpha=0.6, s=s_val*40, label=f"-log10(FDR)={s_val}")
    ax.legend(scatterpoints=1, frameon=True, fontsize=8, loc="lower right")
    plt.tight_layout()
    plt.savefig(f"{OUT}/gsea_dotplot.pdf")
    plt.show()
    print("Saved: gsea_dotplot.pdf")

In [ ]:
# Cell 4 — Enrichment plot for the top-ranked pathway
top_term = gsea_df.iloc[0]["Term"]
print("Top pathway:", top_term)

ax = gp.plot.gseaplot(
    rank_metric=gsea_res.ranking,
    term=top_term,
    **gsea_res.results[top_term]
)
plt.suptitle(f"GSE96870 | {top_term}", y=1.02, fontsize=11)
plt.savefig(f"{OUT}/gsea_top_pathway.pdf", bbox_inches="tight")
plt.show()
print("Saved: gsea_top_pathway.pdf")

---

## Interpreting your results

| Output file | What it shows |
|-------------|---------------|
| `deseq2_results.csv` | Per-gene statistics: log2FC, Wald stat, padj |
| `volcano.pdf` | Significant genes at a glance (red = up in cerebellum, blue = down) |
| `pca.pdf` | Sample clustering — replicates should group by condition |
| `gsea_results.csv` | Per-pathway NES, p-value, FDR, leading-edge genes |
| `gsea_dotplot.pdf` | Pathway landscape (dot size = significance) |
| `gsea_top_pathway.pdf` | Running enrichment score for the top hit |

**Key concepts:**

- **NES > 0** → pathway genes skew toward the top of the ranked list (upregulated in cerebellum)
- **NES < 0** → enriched among downregulated genes (higher in ctrl)
- **FDR q-val < 0.25** is the standard GSEA significance threshold
- The `Lead_genes` column in `gsea_results.csv` lists the core driving genes

---

## Discussion Questions

1. **Filtering:** Why do we pre-filter low-count genes before running DESeq2? What would happen statistically if we skipped this step?
2. **GSEA vs ORA:** Give one scenario where GSEA would reveal a pathway that threshold-based over-representation analysis (ORA) would miss.
3. **Chromosome subset:** This lab used only chr19. What are the trade-offs of this approach, and which gene families or pathways might be under-represented?
4. **Batch effects:** Two of the three ctrl replicates were processed in a different batch. How would you detect this in the PCA?